# 18vB2 — CatBoost quantile temporal OOF predictions

This stage fits the pooled and rule-specific tree comparators on
the same expanding temporal folds as 18vB1.

Each fit uses a deterministic CatBoost `MultiQuantile` objective
at levels \(0.05,0.25,0.50,0.75,0.95\). The five predictions are
monotonised and linearly interpolated onto the frozen 99-point
quantile grid.

Features are limited to deterministic weather information,
calendar position and run timing. No market variable, artificial
ensemble feature or fixed Gaussian bridge is used. The internal
holdout and June external test are not loaded.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Run this notebook from the repository root, not {ROOT}"
    )

UTC = timezone.utc
STEP = "18vB2"
RANDOM_SEED = 20260721
SAMPLE_ORIGIN = pd.Timestamp("2026-03-16")

RULES = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]

A_DIR = (
    ROOT
    / "data/processed/18vA_residual_model_design_and_features"
)
U_DIR = (
    ROOT
    / "data/processed/18uB_model_specific_freeze_support"
)

CANDIDATE_PATH = A_DIR / "18vA_candidate_registry.csv"
QUANTILE_PATH = A_DIR / "18vA_quantile_grid.csv"
DEVELOPMENT_PATH = (
    A_DIR / "18vA_development_labelled_feature_panel.csv"
)
A_SUMMARY_PATH = A_DIR / "18vA_summary.json"
A_MANIFEST_PATH = A_DIR / "18vA_sha256_manifest.csv"

FOLD_TRAIN_PATH = (
    U_DIR / "18uB_fold_training_membership.csv"
)
FOLD_VALIDATION_PATH = (
    U_DIR / "18uB_fold_validation_membership.csv"
)
OOF_SUPPORT_PATH = (
    U_DIR / "18uB_oof_selection_support.csv"
)
U_SUMMARY_PATH = U_DIR / "18uB_summary.json"
U_MANIFEST_PATH = U_DIR / "18uB_sha256_manifest.csv"

OUT_DIR = (
    ROOT
    / "data/processed/18vB2_tree_temporal_oof"
)
REPORT_DIR = (
    ROOT
    / "reports/18vB2_tree_temporal_oof"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = [
    CANDIDATE_PATH,
    QUANTILE_PATH,
    DEVELOPMENT_PATH,
    A_SUMMARY_PATH,
    A_MANIFEST_PATH,
    FOLD_TRAIN_PATH,
    FOLD_VALIDATION_PATH,
    OOF_SUPPORT_PATH,
    U_SUMMARY_PATH,
    U_MANIFEST_PATH,
]

for path in REQUIRED_INPUTS:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified input is missing: {path}"
        )

EXPECTED_CANDIDATES = [
    "catboost_quantile_pooled",
    "catboost_quantile_rule",
]
TREE_QUANTILES = np.array(
    [0.05, 0.25, 0.50, 0.75, 0.95],
    dtype=float,
)
TREE_POOLED_FEATURES = [
    "day_index",
    "forecast_daily_max_c",
    "decision_rule_order",
    "forecast_lead_hours",
    "run_initialisation_hour_utc",
    "day_of_year_sin",
    "day_of_year_cos",
]
TREE_RULE_FEATURES = [
    "day_index",
    "forecast_daily_max_c",
    "forecast_lead_hours",
    "run_initialisation_hour_utc",
    "day_of_year_sin",
    "day_of_year_cos",
]

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(
    series: pd.Series,
    *,
    name: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
    )

    if parsed.isna().any():
        bad = series.loc[
            parsed.isna()
        ].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse Boolean column {name}: {bad}"
        )

    return parsed.astype(bool)


def verify_manifest(path: Path) -> None:
    manifest = pd.read_csv(path)
    failures: list[str] = []

    for row in manifest.itertuples(index=False):
        candidate = ROOT / row.path

        if not candidate.is_file():
            failures.append(f"MISSING: {row.path}")
            continue

        if sha256_file(candidate) != row.sha256:
            failures.append(f"HASH: {row.path}")

        if candidate.stat().st_size != int(row.size_bytes):
            failures.append(f"SIZE: {row.path}")

    if failures:
        raise AssertionError(
            f"Manifest verification failed for {path}:\n"
            + "\n".join(failures)
        )


def add_engineered_features(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    output = frame.copy()
    output["event_date"] = pd.to_datetime(
        output["event_date"],
        errors="raise",
    )

    for column in [
        "decision_cutoff_utc",
        "selected_run_initialisation_utc",
        "selected_run_available_utc",
    ]:
        output[column] = pd.to_datetime(
            output[column],
            utc=True,
            errors="raise",
        )

    if "day_index" not in output.columns:
        output["day_index"] = (
            output["event_date"] - SAMPLE_ORIGIN
        ).dt.days.astype(float)

    if "forecast_lead_hours" not in output.columns:
        output["forecast_lead_hours"] = (
            output["decision_cutoff_utc"]
            - output["selected_run_available_utc"]
        ).dt.total_seconds() / 3600.0

    if (
        "run_initialisation_hour_utc"
        not in output.columns
    ):
        output[
            "run_initialisation_hour_utc"
        ] = output[
            "selected_run_initialisation_utc"
        ].dt.hour.astype(float)

    day_of_year = output[
        "event_date"
    ].dt.dayofyear.astype(float)

    if "day_of_year_sin" not in output.columns:
        output["day_of_year_sin"] = np.sin(
            2.0 * np.pi * day_of_year / 366.0
        )
    if "day_of_year_cos" not in output.columns:
        output["day_of_year_cos"] = np.cos(
            2.0 * np.pi * day_of_year / 366.0
        )

    return output


def interpolate_tree_quantiles(
    trained: np.ndarray,
    output_levels: np.ndarray,
) -> np.ndarray:
    trained = np.maximum.accumulate(
        np.asarray(trained, dtype=float),
        axis=1,
    )

    output = np.vstack(
        [
            np.interp(
                output_levels,
                TREE_QUANTILES,
                row,
                left=row[0],
                right=row[-1],
            )
            for row in trained
        ]
    )

    return np.maximum.accumulate(
        output,
        axis=1,
    )


verify_manifest(A_MANIFEST_PATH)
verify_manifest(U_MANIFEST_PATH)

with A_SUMMARY_PATH.open(encoding="utf-8") as handle:
    a_summary = json.load(handle)
with U_SUMMARY_PATH.open(encoding="utf-8") as handle:
    u_summary = json.load(handle)

if a_summary.get("verdict") != "PASS":
    raise AssertionError("18vA is not a PASS release.")
if u_summary.get("verdict") != "PASS":
    raise AssertionError("18uB is not a PASS release.")

candidates = pd.read_csv(
    CANDIDATE_PATH,
    low_memory=False,
)
candidates = candidates.loc[
    candidates["candidate_id"].isin(
        EXPECTED_CANDIDATES
    )
].copy()

quantile_grid = pd.read_csv(
    QUANTILE_PATH,
    low_memory=False,
)
development = pd.read_csv(
    DEVELOPMENT_PATH,
    low_memory=False,
)
fold_train = pd.read_csv(
    FOLD_TRAIN_PATH,
    low_memory=False,
)
fold_validation = pd.read_csv(
    FOLD_VALIDATION_PATH,
    low_memory=False,
)
oof_support = pd.read_csv(
    OOF_SUPPORT_PATH,
    low_memory=False,
)

for frame in [
    development,
    fold_train,
    fold_validation,
    oof_support,
]:
    frame["event_date"] = pd.to_datetime(
        frame["event_date"],
        errors="raise",
    )

for frame in [
    development,
    fold_train,
    fold_validation,
    oof_support,
]:
    for column in [
        "decision_cutoff_utc",
        "selected_run_initialisation_utc",
        "selected_run_available_utc",
        "current_label_available_utc",
        "model_specific_holdout_freeze_utc",
        "fold_freeze_utc",
    ]:
        if column in frame.columns:
            frame[column] = pd.to_datetime(
                frame[column],
                utc=True,
                errors="raise",
            )

oof_support[
    "freeze_admissible_for_selection"
] = parse_bool(
    oof_support[
        "freeze_admissible_for_selection"
    ],
    name="freeze_admissible_for_selection",
)

development = add_engineered_features(
    development
)
fold_train = add_engineered_features(
    fold_train
)

quantile_levels = quantile_grid[
    "quantile_level"
].to_numpy(dtype=float)
residual_columns = quantile_grid[
    "residual_quantile_column"
].tolist()
hko_columns = quantile_grid[
    "hko_quantile_column"
].tolist()

if list(
    candidates.sort_values(
        "candidate_order"
    )["candidate_id"]
) != EXPECTED_CANDIDATES:
    raise AssertionError(
        "Unexpected tree candidate registry."
    )
if len(development) != 144:
    raise AssertionError(
        f"Development rows: {len(development)}"
    )

print("Verified 18vA and 18uB inputs: PASS")

Verified 18vA and 18uB inputs: PASS


In [3]:
key = ["event_date", "decision_rule"]

def validation_rows(
    scope_id: str,
    fold_id: int,
) -> pd.DataFrame:
    membership = fold_validation.loc[
        fold_validation["scope_id"].eq(scope_id)
        & fold_validation[
            "development_fold"
        ].eq(fold_id)
    ][
        [
            "event_date",
            "decision_rule",
            "development_fold",
        ]
    ]

    result = membership.merge(
        development,
        on=key + ["development_fold"],
        how="left",
        validate="one_to_one",
    )

    if result[
        [
            "forecast_daily_max_c",
            "hko_daily_max_c",
            "residual_c",
        ]
    ].isna().any().any():
        raise AssertionError(
            f"Validation join failed for "
            f"{scope_id}, fold {fold_id}."
        )

    return result


def training_rows(
    scope_id: str,
    fold_id: int,
) -> pd.DataFrame:
    result = fold_train.loc[
        fold_train["scope_id"].eq(scope_id)
        & fold_train[
            "development_fold"
        ].eq(fold_id)
    ].copy()

    if result.empty:
        raise AssertionError(
            f"Empty training set for "
            f"{scope_id}, fold {fold_id}."
        )

    return result


def scope_ids(candidate: dict[str, Any]) -> list[str]:
    if candidate["scope_type"] == "POOLED":
        return ["pooled_all_rules"]

    return [
        f"rule_specific_{rule}"
        for rule in RULES
    ]


def attach_freeze_support(
    predictions: pd.DataFrame,
    candidate: dict[str, Any],
) -> pd.DataFrame:
    if candidate["scope_type"] == "POOLED":
        support = oof_support.loc[
            oof_support["scope_id"].eq(
                "pooled_all_rules"
            )
        ].copy()
    else:
        support = oof_support.loc[
            oof_support["scope_id"].str.startswith(
                "rule_specific_"
            )
        ].copy()

    support = support[
        key
        + [
            "scope_id",
            "freeze_admissible_for_selection",
            "model_specific_holdout_freeze_utc",
        ]
    ]

    return predictions.merge(
        support,
        on=key + ["scope_id"],
        how="left",
        validate="one_to_one",
    )


prediction_frames = []
diagnostic_rows = []

alpha_text = ",".join(
    f"{alpha:.2f}"
    for alpha in TREE_QUANTILES
)

for candidate in candidates.sort_values(
    "candidate_order"
).to_dict(orient="records"):
    candidate_id = candidate["candidate_id"]

    for scope_id in scope_ids(candidate):
        for fold_id in [1, 2, 3, 4]:
            train = training_rows(
                scope_id,
                fold_id,
            )
            valid = validation_rows(
                scope_id,
                fold_id,
            )

            feature_columns = (
                TREE_POOLED_FEATURES
                if candidate_id
                == "catboost_quantile_pooled"
                else TREE_RULE_FEATURES
            )

            x_train = train[
                feature_columns
            ].to_numpy(dtype=float)
            x_valid = valid[
                feature_columns
            ].to_numpy(dtype=float)
            y_train = train[
                "residual_c"
            ].to_numpy(dtype=float)
            weights = train[
                "date_balanced_training_weight"
            ].to_numpy(dtype=float)

            model = CatBoostRegressor(
                loss_function=(
                    "MultiQuantile:alpha="
                    + alpha_text
                ),
                iterations=250,
                depth=3,
                learning_rate=0.03,
                l2_leaf_reg=8.0,
                random_seed=RANDOM_SEED,
                bootstrap_type="No",
                random_strength=0.0,
                verbose=False,
                allow_writing_files=False,
                thread_count=1,
            )
            model.fit(
                x_train,
                y_train,
                sample_weight=weights,
                verbose=False,
            )

            trained = np.asarray(
                model.predict(x_valid),
                dtype=float,
            )

            if trained.ndim == 1:
                trained = trained[:, None]

            if trained.shape != (
                len(valid),
                len(TREE_QUANTILES),
            ):
                raise AssertionError(
                    "Unexpected MultiQuantile shape: "
                    f"{trained.shape}"
                )

            quantiles = interpolate_tree_quantiles(
                trained,
                quantile_levels,
            )

            output = valid[
                [
                    "event_date",
                    "decision_rule",
                    "decision_rule_order",
                    "development_fold",
                    "forecast_daily_max_c",
                    "hko_daily_max_c",
                    "residual_c",
                    "current_label_available_utc",
                ]
            ].copy()
            output["candidate_id"] = candidate_id
            output["model_family"] = "TREE"
            output["scope_type"] = candidate[
                "scope_type"
            ]
            output["scope_id"] = scope_id
            output["complexity_rank"] = int(
                candidate["complexity_rank"]
            )

            for column, values in zip(
                residual_columns,
                quantiles.T,
            ):
                output[column] = values

            hko_quantiles = (
                output[
                    "forecast_daily_max_c"
                ].to_numpy(dtype=float)[:, None]
                + quantiles
            )
            for column, values in zip(
                hko_columns,
                hko_quantiles.T,
            ):
                output[column] = values

            output["predicted_residual_mean_c"] = (
                quantiles.mean(axis=1)
            )
            output[
                "predicted_residual_median_c"
            ] = quantiles[:, 49]
            output["prediction_status"] = "PASS"

            output = attach_freeze_support(
                output,
                candidate,
            )

            if output[
                "freeze_admissible_for_selection"
            ].isna().any():
                raise AssertionError(
                    "Missing freeze support."
                )

            prediction_frames.append(output)

            feature_importance = (
                model.get_feature_importance()
            )
            diagnostic_rows.append(
                {
                    "candidate_id": candidate_id,
                    "scope_id": scope_id,
                    "development_fold": fold_id,
                    "training_rows": len(train),
                    "training_dates": train[
                        "event_date"
                    ].nunique(),
                    "validation_rows": len(valid),
                    "validation_dates": valid[
                        "event_date"
                    ].nunique(),
                    "fit_status": "PASS",
                    "fitted_parameters": json.dumps(
                        {
                            "loss": (
                                "MultiQuantile"
                            ),
                            "training_quantiles": (
                                TREE_QUANTILES.tolist()
                            ),
                            "iterations": 250,
                            "depth": 3,
                            "learning_rate": 0.03,
                            "l2_leaf_reg": 8.0,
                            "features": feature_columns,
                            "feature_importance": {
                                feature: float(value)
                                for feature, value in zip(
                                    feature_columns,
                                    feature_importance,
                                )
                            },
                        }
                    ),
                }
            )

predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)
diagnostics = pd.DataFrame(
    diagnostic_rows
)

if len(predictions) != 2 * 144:
    raise AssertionError(
        f"Expected 288 predictions, "
        f"found {len(predictions)}."
    )

counts = predictions.groupby(
    "candidate_id"
).size()
if not counts.eq(144).all():
    raise AssertionError(
        f"Candidate coverage differs:\n{counts}"
    )

if predictions.duplicated(
    [
        "candidate_id",
        "event_date",
        "decision_rule",
    ]
).any():
    raise AssertionError(
        "Duplicate candidate-date-rule predictions."
    )

if not (
    np.diff(
        predictions[
            residual_columns
        ].to_numpy(dtype=float),
        axis=1,
    )
    >= -1e-12
).all():
    raise AssertionError(
        "Non-monotone tree predictive quantiles."
    )

print("Tree temporal OOF predictions: PASS")
print(f"Prediction rows: {len(predictions):,}")
display(diagnostics)

/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

Tree temporal OOF predictions: PASS
Prediction rows: 288


/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99245/2086033113.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

,candidate_id,scope_id,development_fold,training_rows,training_dates,validation_rows,validation_dates,fit_status,fitted_parameters
0,catboost_quantile_pooled,pooled_all_rules,1,56,22,32,10,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
1,catboost_quantile_pooled,pooled_all_rules,2,96,32,40,10,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
2,catboost_quantile_pooled,pooled_all_rules,3,136,42,36,9,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
3,catboost_quantile_pooled,pooled_all_rules,4,168,50,36,9,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
4,catboost_quantile_rule,rule_specific_24h_prior,1,16,16,10,10,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
5,catboost_quantile_rule,rule_specific_24h_prior,2,26,26,10,10,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
6,catboost_quantile_rule,rule_specific_24h_prior,3,36,36,9,9,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
7,catboost_quantile_rule,rule_specific_24h_prior,4,44,44,9,9,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
8,catboost_quantile_rule,rule_specific_12h_prior,1,16,16,7,7,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
9,catboost_quantile_rule,rule_specific_12h_prior,2,23,23,10,10,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."


In [4]:
check_rows = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
) -> None:
    check_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": True,
        }
    )

add_check(
    "candidate_models_2",
    len(candidates) == 2,
    f"candidates={len(candidates)}",
)
add_check(
    "prediction_rows_288",
    len(predictions) == 288,
    f"rows={len(predictions)}",
)
add_check(
    "each_candidate_has_144_rows",
    counts.eq(144).all(),
    counts.to_dict().__str__(),
)
add_check(
    "diagnostic_rows_20",
    len(diagnostics) == 20,
    f"rows={len(diagnostics)}",
)
add_check(
    "quantiles_monotone",
    (
        np.diff(
            predictions[
                residual_columns
            ].to_numpy(dtype=float),
            axis=1,
        )
        >= -1e-12
    ).all(),
    "99 residual quantiles",
)
add_check(
    "development_only",
    predictions["event_date"].le(
        pd.Timestamp("2026-05-21")
    ).all(),
    "no holdout or June row",
)
add_check(
    "all_prediction_status_pass",
    predictions[
        "prediction_status"
    ].eq("PASS").all(),
    "no fit failure",
)
add_check(
    "market_information_absent",
    "p_market" not in predictions.columns,
    "weather residual candidates only",
)
add_check(
    "fixed_bridge_absent",
    not any(
        (
            "gaussian_bridge" in column.lower()
            or "ecmwf_proxy" in column.lower()
        )
        for column in predictions.columns
    ),
    "no archived bridge field",
)

integrity = pd.DataFrame(check_rows)
if not integrity["passed"].all():
    raise AssertionError(
        "18vB2 blocking checks failed:\n"
        + integrity.loc[
            ~integrity["passed"]
        ].to_string(index=False)
    )

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "candidate_id",
        "scope_id",
        "development_fold",
        "event_date",
        "decision_rule",
        "detail",
        "blocking",
    ]
)

print("18vB2 integrity checks: PASS")

18vB2 integrity checks: PASS


In [5]:
prediction_path = (
    OUT_DIR / "18vB2_tree_oof_predictions.csv"
)
diagnostics_path = (
    OUT_DIR / "18vB2_model_fit_diagnostics.csv"
)
integrity_path = (
    OUT_DIR / "18vB2_integrity_checks.csv"
)
issues_path = OUT_DIR / "18vB2_issues.csv"

output_predictions = predictions.copy()
output_diagnostics = diagnostics.copy()

for frame in [
    output_predictions,
    output_diagnostics,
]:
    for column in frame.columns:
        if "date" in column.lower():
            if pd.api.types.is_datetime64_any_dtype(
                frame[column]
            ):
                frame[column] = frame[
                    column
                ].dt.strftime("%Y-%m-%d")

        if (
            "freeze" in column.lower()
            or "cutoff" in column.lower()
            or column.lower().endswith("_utc")
            or "available" in column.lower()
        ):
            frame[column] = frame[
                column
            ].astype("string")

output_predictions.to_csv(
    prediction_path,
    index=False,
)
output_diagnostics.to_csv(
    diagnostics_path,
    index=False,
)
integrity.to_csv(
    integrity_path,
    index=False,
)
issues.to_csv(
    issues_path,
    index=False,
)

source_inventory = pd.DataFrame(
    [
        {
            "input_role": "18vA_candidate_registry",
            "path": str(
                CANDIDATE_PATH.relative_to(ROOT)
            ),
            "rows": len(candidates),
            "sha256": sha256_file(
                CANDIDATE_PATH
            ),
        },
        {
            "input_role": "18vA_quantile_grid",
            "path": str(
                QUANTILE_PATH.relative_to(ROOT)
            ),
            "rows": len(quantile_grid),
            "sha256": sha256_file(
                QUANTILE_PATH
            ),
        },
        {
            "input_role": "18vA_development_panel",
            "path": str(
                DEVELOPMENT_PATH.relative_to(ROOT)
            ),
            "rows": len(development),
            "sha256": sha256_file(
                DEVELOPMENT_PATH
            ),
        },
        {
            "input_role": "18uB_fold_training_membership",
            "path": str(
                FOLD_TRAIN_PATH.relative_to(ROOT)
            ),
            "rows": len(fold_train),
            "sha256": sha256_file(
                FOLD_TRAIN_PATH
            ),
        },
        {
            "input_role": "18uB_fold_validation_membership",
            "path": str(
                FOLD_VALIDATION_PATH.relative_to(ROOT)
            ),
            "rows": len(fold_validation),
            "sha256": sha256_file(
                FOLD_VALIDATION_PATH
            ),
        },
        {
            "input_role": "18uB_oof_selection_support",
            "path": str(
                OOF_SUPPORT_PATH.relative_to(ROOT)
            ),
            "rows": len(oof_support),
            "sha256": sha256_file(
                OOF_SUPPORT_PATH
            ),
        },
    ]
)
source_inventory_path = (
    OUT_DIR / "18vB2_source_inventory.csv"
)
source_inventory.to_csv(
    source_inventory_path,
    index=False,
)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "candidate_models": int(len(candidates)),
    "temporal_oof_prediction_rows": int(
        len(predictions)
    ),
    "rows_per_candidate": 144,
    "model_fit_diagnostic_rows": int(
        len(diagnostics)
    ),
    "catboost_loss": "MultiQuantile",
    "tree_training_quantiles": (
        TREE_QUANTILES.tolist()
    ),
    "holdout_used": False,
    "external_test_used": False,
    "market_information_used": False,
    "artificial_ensemble_features_used": False,
    "fixed_gaussian_bridge_used": False,
    "issue_rows": 0,
    "integrity_checks_passed": int(
        integrity["passed"].sum()
    ),
    "integrity_checks_total": int(
        len(integrity)
    ),
}

summary_path = OUT_DIR / "18vB2_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "catboost": __import__("catboost").__version__,
    "revision": "v1",
}
environment_path = (
    OUT_DIR / "18vB2_environment.json"
)
environment_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# 18vB2 CatBoost temporal OOF predictions",
    "",
    "**PASS**",
    "",
    f"- Candidates: {len(candidates):,}",
    f"- Prediction rows: {len(predictions):,}",
    "- OOF rows per candidate: 144",
    (
        f"- Scope-fold fits: "
        f"{len(diagnostics):,}"
    ),
    "",
    (
        "Each fit uses one deterministic MultiQuantile "
        "model at 0.05, 0.25, 0.50, 0.75 and 0.95."
    ),
    (
        "No holdout or external row, market feature, "
        "artificial ensemble feature or fixed Gaussian "
        "bridge is used."
    ),
]

report_path = (
    REPORT_DIR
    / "18vB2_tree_temporal_oof_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18vB2_sha256_manifest.csv":
            continue

        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = (
    OUT_DIR / "18vB2_sha256_manifest.csv"
)
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2))
print("18vB2 tree OOF release: PASS")

{
  "step": "18vB2",
  "generated_at_utc": "2026-07-21T22:43:06.906821+00:00",
  "verdict": "PASS",
  "candidate_models": 2,
  "temporal_oof_prediction_rows": 288,
  "rows_per_candidate": 144,
  "model_fit_diagnostic_rows": 20,
  "catboost_loss": "MultiQuantile",
  "tree_training_quantiles": [
    0.05,
    0.25,
    0.5,
    0.75,
    0.95
  ],
  "holdout_used": false,
  "external_test_used": false,
  "market_information_used": false,
  "artificial_ensemble_features_used": false,
  "fixed_gaussian_bridge_used": false,
  "issue_rows": 0,
  "integrity_checks_passed": 9,
  "integrity_checks_total": 9
}
18vB2 tree OOF release: PASS
